In [19]:
import numpy as np
from embedder import Embedder
from ingest import load_faq_data
from tqdm.auto import tqdm

In [2]:
documents = load_faq_data()

In [3]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [20]:
embed = Embedder()

In [5]:
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [6]:
X.shape

(1350, 384)

In [21]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

In [8]:
scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'course': 'llm-zoomcamp',
 'section': 'Module 2: Vector Search',
 'question': 'Why does cosine similarity reduce to a matrix multiplication between the embeddings and the query vector?',
 'answer': 'Cosine similarity measures how aligned two vectors are, regardless of their magnitude. When all vectors (including the query) are normalized to unit length, their magnitudes no longer matter. In this case, cosine similarity is equivalent to simply taking the dot product between the query and each document embedding. This allows us to compute similarities efficiently using matrix multiplication.',
 'doc_id': '95feb4e75b'}

# Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

- -0.31
- -0.02   <------------
- 0.12
- 0.44

In [10]:
v_query[0]

np.float64(-0.02058203437252893)

In [22]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
len(documents)

72

# Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

- 0.07
- 0.37   <------------
- 0.68
- 0.92

In [25]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(d for d in documents if d["filename"] == target_filename)
v_doc = embed.encode(doc["content"])

# Vectores normalizados => dot product == cosine similarity
cosine_similarity = float(v_query.dot(v_doc))
cosine_similarity

0.36107027225589694

# Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its filename)?

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- 02-vector-search/lessons/07-sqlitesearch-vector.md   <------------
- 02-vector-search/lessons/09-onnx-embedder.md

In [27]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [30]:
batch_size = 50
chunk_texts = [chunk["content"] for chunk in chunks]

chunk_vectors = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    chunk_vectors.extend(batch_vectors)

X = np.vstack(chunk_vectors)
scores = X.dot(v_query)
scores.shape

  0%|          | 0/6 [00:00<?, ?it/s]

(295,)

In [31]:
best_idx = int(np.argmax(scores))
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- 04-evaluation/lessons/05-search-metrics.md   <------------
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

In [32]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)
results = index.search(v_query, num_results=5)

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search

 Which file shows up in the vector results but not in the text results?

- 02-vector-search/lessons/01-intro.md
- 02-vector-search/lessons/02-embeddings.md
- 02-vector-search/lessons/08-pgvector.md   <------------
- 03-orchestration/lessons/05-rag.md

In [33]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"] )
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"
v_query = embed.encode(query)

vector_results = index.search(v_query, num_results=5)
text_results = text_index.search(query, num_results=5)

vector_files = [r["filename"] for r in vector_results]
text_files = [r["filename"] for r in text_results]

vector_only = list(set(vector_files) - set(text_files))
vector_only, vector_files, text_files

(['02-vector-search/lessons/08-pgvector.md'],
 ['02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md'],
 ['02-vector-search/lessons/02-embeddings.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md'])

# Q6. Hybrid search

Which file is ranked first after RRF?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/13-function-calling.md   <------------
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md

In [34]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [35]:
query = "How do I give the model access to tools?"
v_query = embed.encode(query)

vector_results = index.search(v_query, num_results=5)
text_results = text_index.search(query, num_results=5)

hybrid_results = rrf([vector_results, text_results], k=60, num_results=5)

[(doc["filename"], doc["start"]) for doc in hybrid_results]

[('01-agentic-rag/lessons/13-function-calling.md', 4000),
 ('01-agentic-rag/lessons/01-intro.md', 2000),
 ('01-agentic-rag/lessons/14-agentic-loop.md', 0),
 ('04-evaluation/lessons/02-ground-truth.md', 1000),
 ('01-agentic-rag/lessons/16-other-frameworks.md', 0)]